# K-pop 뮤직비디오 유튜브 성과 데이터 수집

이 노트북은 K-pop 뮤직비디오의 유튜브 성과 데이터를 모으기 위한 크롤링 파이프라인입니다. 크게 두 단계로 구성됩니다.

1. **월간 순위 리스트 수집** (셀 1) — 음악 순위 집계 사이트의 API에서 2023~2025년 월간 유튜브 순위 데이터를 가져옵니다.
2. **유튜브 메타데이터 보강** (셀 2) — 1단계에서 수집한 영상 URL로 YouTube Data API를 호출해 조회수, 좋아요, 댓글 수 등의 최신 지표를 붙입니다.

**실행 전 참고사항**
- `base_url`은 실제 엔드포인트를 가리지 않기 위해 예시 도메인(`target_site.com`)으로 표시해 두었습니다. 직접 실행하시려면 본인이 사용하는 실제 API 엔드포인트로 교체해야 합니다.
- YouTube Data API 키는 코드에 직접 넣지 않고, 환경변수 `YOUTUBE_API_KEY`에 등록해서 사용합니다.
- 수집된 원본 데이터(CSV/JSON)에는 곡명·아티스트·썸네일 URL 등 저작권이 있는 정보가 포함되어 있어, 이 저장소에는 코드만 공개하고 원본 데이터 파일은 포함하지 않습니다.


## 1. 월간 순위 리스트 크롤링

음악 순위 집계 사이트의 API(`/api/youtube/monthlyData`)에서 2023~2025년, 1~12월 유튜브 순위 데이터를 페이지네이션(`lastOrderNo`)으로 끝까지 수집합니다.

- 곡명, 아티스트, 재생수, 순위, 유튜브 URL 등을 담은 리스트를 만듭니다.
- 결과를 `kpop_radar_2023_2025_full.csv` / `.json`으로 저장합니다.


<h1>2023~2025년 월간 유튜브 순위 리스트를 크롤링해서 CSV/JSON으로 저장


In [ ]:
import requests
import time
import pandas as pd
import json

base_url = "https://target_site.com/api/youtube/monthlyData"
years = ['2025', '2024', '2023']
year_label = f"{min(years)}_{max(years)}"
all_monthly_data = []

for year in years:
    for month in range(1, 13):
        print(f"\n📅 {year}년 {month}월 수집 중...")
        month_count = 0
        
        for last_no in range(0, 600, 50):
            params = {
                "sortType": "1",
                "dateOrder": "1",
                "orderCountInPage": "50",
                "lastOrderNo": str(last_no),
                "year": year,
                "month": str(month)
            }
            
            try:
                response = requests.get(base_url, params=params, timeout=10)
                
                if response.status_code == 200:
                    full_json = response.json()
                    
                    # ✅ 수정된 부분: data.tasks로 접근
                    page_items = full_json.get('data', {}).get('tasks', [])
                    
                    if isinstance(page_items, list) and len(page_items) > 0:
                        for item in page_items:
                            if isinstance(item, dict):
                                item['view_year'] = year
                                item['view_month'] = month
                                item['lastOrderNo'] = last_no
                                all_monthly_data.append(item)
                                month_count += 1
                        
                        print(f"  └ lastOrderNo={last_no}: {len(page_items)}개 수집 (누적: {month_count}개)", end="\r")
                        
                    elif last_no == 0:  # 첫 페이지에서도 데이터 없으면
                        print(f"  ⚠️ {year}년 {month}월은 데이터가 없습니다. (아직 발표 안됨)")
                        break
                
                elif response.status_code == 404:
                    break  # 더 이상 페이지 없음
                
                time.sleep(0.3)
                
            except Exception as e:
                print(f"\n❌ 에러: {e}")
                continue
    
    print(f"\n✅ {month}월 완료: {month_count}개")

# ============================================
# 결과 저장
# ============================================
if all_monthly_data:
    df = pd.DataFrame(all_monthly_data)
    
    print(f"\n{'='*60}")
    print(f"📊 최종 수집 결과")
    print(f"{'='*60}")
    print(f"총 행 수: {len(df):,}개")
    print(f"컬럼 수: {len(df.columns)}개")
    
    print(f"\n📋 컬럼 목록:")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:2d}. {col}")
    
    # CSV 저장
    df.to_csv(f'kpop_radar_{year_label}_full.csv', index=False, encoding='utf-8-sig')
    
    # JSON 백업
    with open(f'kpop_radar_{year_label}_full.json', 'w', encoding='utf-8') as f:
        json.dump(all_monthly_data, f, ensure_ascii=False, indent=2)
    
    # 미리보기
    print(f"\n{'='*60}")
    print("📋 데이터 샘플 (처음 5개)")
    print(f"{'='*60}")
    
    # 주요 컬럼만 보기
    display_cols = ['songName', 'artists', 'playCount', 'incCount', 'orderNo', 'publishTime', 'view_month']
    display_cols = [col for col in display_cols if col in df.columns]
    print(df[display_cols].head(5).to_string(index=False))
    
    print(f"\n✅ 파일 저장 완료:")
    print(f"  - kpop_radar_{year_label}_full.csv ({len(df):,}행)")
    print(f"  - kpop_radar_{year_label}_full.json")
    
else:
    print("\n❌ 데이터가 수집되지 않았습니다.")

## 2. 유튜브 메타데이터 보강

셀 1에서 저장한 CSV를 불러와 URL에서 `video_id`를 추출하고, 중복 제거한 고유 영상 ID를 50개씩 묶어 YouTube Data API(`videos` 엔드포인트)에 조회합니다.

- 조회수, 좋아요 수, 댓글 수, 게시일, 영상 길이, 제목, 채널명을 가져옵니다.
- 원본 데이터와 `video_id` 기준으로 병합해 `kpop_radar_2023_2025_full_with_youtubeapi.csv`로 저장합니다.


In [ ]:
import os
import pandas as pd
import requests
import time
from urllib.parse import urlparse, parse_qs
import json

# ============================================
# STEP 1: YouTube API 설정
# ============================================
YOUTUBE_API_KEY = os.environ["YOUTUBE_API_KEY"]  # 환경변수 YOUTUBE_API_KEY로 등록해서 사용

def extract_video_id(url):
    """YouTube URL에서 video_id 추출"""
    try:
        parsed_url = urlparse(url)
        
        # youtu.be 형식
        if 'youtu.be' in parsed_url.netloc:
            return parsed_url.path[1:]
        
        # youtube.com/watch?v= 형식
        elif 'youtube.com' in parsed_url.netloc:
            query_params = parse_qs(parsed_url.query)
            return query_params.get('v', [None])[0]
        
        return None
    except:
        return None

def get_youtube_data_batch(video_ids, api_key):
    """
    YouTube Data API로 비디오 정보 가져오기 (최대 50개씩)
    """
    base_url = "https://www.googleapis.com/youtube/v3/videos"
    
    params = {
        'key': api_key,
        'part': 'snippet,statistics,contentDetails',
        'id': ','.join(video_ids)  # 최대 50개
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        
        if response.status_code == 200:
            return response.json()
        else:
            print(f"❌ API 에러 (status {response.status_code}): {response.text[:200]}")
            return None
            
    except Exception as e:
        print(f"❌ 요청 실패: {e}")
        return None

def parse_youtube_response(video_id, item):
    """API 응답을 파싱해서 딕셔너리로 변환"""
    try:
        snippet = item.get('snippet', {})
        statistics = item.get('statistics', {})
        content_details = item.get('contentDetails', {})
        
        return {
            'video_id': video_id,
            'api_view_count': int(statistics.get('viewCount', 0)),
            'api_like_count': int(statistics.get('likeCount', 0)),
            'api_comment_count': int(statistics.get('commentCount', 0)),
            'api_published_at': snippet.get('publishedAt', ''),
            'api_duration': content_details.get('duration', ''),
            'api_title': snippet.get('title', ''),
            'api_channel_title': snippet.get('channelTitle', '')
        }
    except Exception as e:
        print(f"⚠️ 파싱 에러 (video_id={video_id}): {e}")
        return {
            'video_id': video_id,
            'api_view_count': 0,
            'api_like_count': 0,
            'api_comment_count': 0,
            'api_published_at': '',
            'api_duration': '',
            'api_title': '',
            'api_channel_title': ''
        }

# ============================================
# STEP 2: 크롤링한 데이터 로드
# ============================================
print("📂 데이터 로딩 중...")
df = pd.read_csv(r'C:\Users\sy-77\Downloads\kpop_radar_2023_2025_full.csv', encoding='utf-8-sig')

print(f"✅ 총 {len(df):,}개 로드")
print(f"컬럼: {', '.join(df.columns.tolist())}")

# ============================================
# STEP 3: video_id 추출
# ============================================
print("\n🔗 video_id 추출 중...")
df['video_id'] = df['url'].apply(extract_video_id)

# video_id 없는 행 제거
df_valid = df[df['video_id'].notna()].copy()
print(f"✅ 유효한 video_id: {len(df_valid):,}개 (제외: {len(df) - len(df_valid)}개)")

# 중복 제거 (같은 비디오가 여러 달 나올 수 있음)
video_ids_unique = df_valid['video_id'].unique()
print(f"✅ 고유 video_id: {len(video_ids_unique):,}개")

# ============================================
# STEP 4: YouTube API 호출 (50개씩 배치)
# ============================================
print(f"\n🎬 YouTube API 호출 시작...")
print(f"예상 API 호출 횟수: {(len(video_ids_unique) + 49) // 50}회")
print(f"예상 소요 시간: 약 {(len(video_ids_unique) + 49) // 50 * 2}초\n")

youtube_data = {}
batch_size = 50

for i in range(0, len(video_ids_unique), batch_size):
    batch = video_ids_unique[i:i+batch_size].tolist()
    batch_num = i // batch_size + 1
    total_batches = (len(video_ids_unique) + batch_size - 1) // batch_size
    
    print(f"[{batch_num}/{total_batches}] {len(batch)}개 비디오 조회 중...", end=" ")
    
    # API 호출
    response_data = get_youtube_data_batch(batch, YOUTUBE_API_KEY)
    
    if response_data:
        items = response_data.get('items', [])
        
        # 응답 파싱
        for item in items:
            video_id = item.get('id')
            parsed = parse_youtube_response(video_id, item)
            youtube_data[video_id] = parsed
        
        print(f"✅ {len(items)}개 성공")
        
        # 못 가져온 비디오 (삭제됨/비공개 등)
        missing = set(batch) - set([item.get('id') for item in items])
        if missing:
            print(f"  ⚠️ 누락: {len(missing)}개 (삭제/비공개 등)")
            for vid in missing:
                youtube_data[vid] = {
                    'video_id': vid,
                    'api_view_count': 0,
                    'api_like_count': 0,
                    'api_comment_count': 0,
                    'api_published_at': '',
                    'api_duration': '',
                    'api_title': '',
                    'api_channel_title': ''
                }
    
    # Rate limiting (1초 대기)
    time.sleep(1)

# ============================================
# STEP 5: 데이터프레임에 병합
# ============================================
print(f"\n📊 데이터 병합 중...")

# YouTube 데이터를 데이터프레임으로 변환
df_youtube = pd.DataFrame(youtube_data.values())

# 원본 데이터와 병합
df_final = df_valid.merge(df_youtube, on='video_id', how='left')

print(f"✅ 병합 완료: {len(df_final):,}행")

# ============================================
# STEP 6: 결과 저장
# ============================================
output_file = r"C:\Users\sy-77\Downloads\kpop_radar_2023_2025_full_with_youtubeapi.csv"
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n{'='*60}")
print(f"📊 최종 결과")
print(f"{'='*60}")
print(f"총 행 수: {len(df_final):,}개")
print(f"컬럼 수: {len(df_final.columns)}개")

print(f"\n📋 컬럼 목록:")
for i, col in enumerate(df_final.columns, 1):
    null_count = df_final[col].isna().sum()
    print(f"  {i:2d}. {col:25s} - {len(df_final) - null_count:,} non-null")

print(f"\n📈 통계:")
print(f"  • 평균 조회수: {df_final['api_view_count'].mean():,.0f}")
print(f"  • 평균 좋아요: {df_final['api_like_count'].mean():,.0f}")
print(f"  • 평균 댓글: {df_final['api_comment_count'].mean():,.0f}")

print(f"\n📋 샘플 데이터 (처음 3개):")
display_cols = ['songName', 'artists', 'api_view_count', 'api_like_count', 'api_title']
display_cols = [col for col in display_cols if col in df_final.columns]
print(df_final[display_cols].head(3).to_string(index=False))

print(f"\n✅ 파일 저장: {output_file}")